# Part1

# Task1

1. What is a tool in an AI Agent?
A tool is an external function or capability (API, calculator, database, web search, etc.) that an AI agent can call to perform actions beyond text generation.

2. Why do agents need tools?
LLMs alone cannot:

Do accurate math

Fetch real-time data

Access databases or APIs
Tools extend LLMs with action + execution power.

3. Difference between a chatbot and an agent

Chatbot only responds with text, no external actions, stateless, passive
Agent can think + act, uses tool, goal-driven, autonomous


# Task2

In [1]:
pip install langchain langchain-community langchain-groq wikipedia tavily-python


Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    model_name="llama3-8b-8192"
)


In [10]:
from langchain_community.tools import (
    WikipediaQueryRun,
    DuckDuckGoSearchRun
)
from langchain_community.utilities import WikipediaAPIWrapper
import math


In [12]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Performs mathematical calculations"""
    return str(eval(expression))


In [13]:
wiki = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)


In [14]:
search = DuckDuckGoSearchRun()


In [17]:
print("Calculator Output:", calculator.run("25 * 4 + 10"))
print("Wikipedia Output:", wiki.run("LangChain"))
print("Web Search Output:", search.run("Latest AI trends"))


Calculator Output: 110
Wikipedia Output: Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Model Context Protocol
Summary: The Model Context Protocol (MCP) is an open standard and open-source framework introduced by Anthropic in November 2024 to standardize the way artificial intelligence (AI) systems like large language models (LLMs) integrate and share data with external tools, systems, and data sources. MCP provides a universal interface for reading files, executing functions, and handling contextual prompts. Following its announcement, the protocol was adopted by major AI providers, including OpenAI and Google DeepMind.



Page: AI agent
Summary: In the context of generative artifi

# Part2

# Task3

In [18]:
from langchain.tools import tool

@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information"""
    policies = {
        "leave": "Employees are entitled to 20 paid leaves per year.",
        "remote": "Remote work allowed up to 3 days per week.",
        "security": "2FA is mandatory for all accounts."
    }
    return policies.get(query.lower(), "Policy not found")


In [20]:
print(company_policy_lookup.run("leave"))


Employees are entitled to 20 paid leaves per year.


# Task4

In [21]:
from datetime import datetime

@tool
def current_datetime() -> str:
    """Returns current date and time"""
    return datetime.now().isoformat()

@tool
def simple_db_query(name: str) -> str:
    """Mock database lookup"""
    db = {"alice": "Engineer", "bob": "Manager"}
    return db.get(name.lower(), "User not found")


In [22]:
custom_toolkit = [
    company_policy_lookup,
    current_datetime,
    simple_db_query
]


# Part3

# Task5

In [47]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

# LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant"
)

# Tools
@tool
def calculator(expression: str) -> str:
    """Performs mathematical calculations"""
    return str(eval(expression))

@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information"""
    policies = {
        "leave": "Employees get 20 paid leaves per year",
        "remote": "Remote work allowed 3 days a week"
    }
    return policies.get(query.lower(), "Policy not found")

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)

# Bind tools to LLM  ✅ TASK 5 COMPLETE
llm_with_tools = llm.bind_tools(
    [calculator, company_policy_lookup, wiki_tool]
)


# Task6

In [48]:
from langchain_core.messages import ToolMessage

# User query
messages = [
    HumanMessage(content="What is 25 * 4 + 10?")
]

# LLM decides which tool to call
response = llm_with_tools.invoke(messages)

# Execute the tool chosen by LLM
tool_calls = response.tool_calls
tool_outputs = []

for call in tool_calls:
    if call["name"] == "calculator":
        result = calculator.invoke(call["args"])
    elif call["name"] == "company_policy_lookup":
        result = company_policy_lookup.invoke(call["args"])
    elif call["name"] == "WikipediaQueryRun":
        result = wiki_tool.run(call["args"]["query"])
    else:
        result = "Unknown tool"

    tool_outputs.append(
        ToolMessage(
            tool_call_id=call["id"],
            content=str(result)
        )
    )

# Final answer from LLM
final_response = llm_with_tools.invoke(
    messages + [response] + tool_outputs
)

print(final_response.content)


The result of the calculation is 110.


# Part4

# Task7

 Task 7: ReAct Overview (Conceptual)

ReAct = Reason + Act

The agent reasons about the problem

Decides which tool to use

Executes the tool

Observes result

Produces final answer

Why ReAct is powerful

Transparent reasoning

Dynamic tool usage

Handles multi-step tasks

# Task8

In [49]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(
        content="""
        You are an AI assistant that follows ReAct reasoning.
        Think step by step and decide which tool to use.
        Question: What is LangChain?
        """
    )
]

response = llm_with_tools.invoke(messages)

print("LLM Initial Response:")
print(response.content)
print("\nTool Calls Detected:")
print(response.tool_calls)


LLM Initial Response:
<wikipedia>{"query": "LangChain"}</wikipedia>

Tool Calls Detected:
[]


# Task9

In [50]:
from langchain_core.messages import ToolMessage

messages = [
    HumanMessage(
        content="What is LangChain and how many paid leaves do employees get?"
    )
]

# Step 1: LLM decides tools
response = llm_with_tools.invoke(messages)

tool_outputs = []

for call in response.tool_calls:
    if call["name"] == "WikipediaQueryRun":
        result = wiki_tool.run(call["args"]["query"])
    elif call["name"] == "company_policy_lookup":
        result = company_policy_lookup.invoke(call["args"])
    else:
        result = "Unknown tool"

    tool_outputs.append(
        ToolMessage(
            tool_call_id=call["id"],
            content=str(result)
        )
    )

# Step 2: Final reasoning + answer
final_response = llm_with_tools.invoke(
    messages + [response] + tool_outputs
)

print(final_response.content)


# Part5

# Task10

In [51]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        content="""
        Explain what AI agents are,
        calculate 15 * 12,
        and tell me how many paid leaves employees get.
        """
    )
]

response = llm_with_tools.invoke(messages)

tool_outputs = []

for call in response.tool_calls:
    if call["name"] == "calculator":
        result = calculator.invoke(call["args"])
    elif call["name"] == "company_policy_lookup":
        result = company_policy_lookup.invoke(call["args"])
    elif call["name"] == "WikipediaQueryRun":
        result = wiki_tool.run(call["args"]["query"])
    else:
        result = "Unknown tool"

    tool_outputs.append(
        ToolMessage(
            tool_call_id=call["id"],
            content=str(result)
        )
    )

final_response = llm_with_tools.invoke(
    messages + [response] + tool_outputs
)

print("Final Assistant Output:\n")
print(final_response.content)


Final Assistant Output:

Since the policy was not found, I will provide some general information about paid leaves.

The number of paid leaves that employees get can vary depending on the company, location, and industry. In many countries, employees are entitled to a certain number of paid leaves per year, such as vacation days, sick leave, and holidays. Additionally, some companies may offer additional paid leaves, such as parental leave or bereavement leave.

If you are looking for specific information about a particular company or location, I recommend checking the company's HR policies or contacting HR directly for more information.


# Task11

1. Benefits of Tool-Augmented Agents

Accurate calculations

Real-time data

Autonomous decision-making

2. Challenges

Tool errors

Cost

Latency

Prompt tuning